# 02 - Uczenie Modelu Segmentacji

Ten notebook zawiera trening segmentation U-Net na danych z katalogu segmentation/train.

In [ ]:
import time
from pathlib import Path
import numpy as np
import torch

PROJECT_ROOT = Path.cwd().resolve()
import sys
if not (PROJECT_ROOT / 'src').exists():
    p = PROJECT_ROOT
    while True:
        if (p / 'src').exists():
            PROJECT_ROOT = p
            break
        if p == p.parent:
            break
        p = p.parent
sys.path.insert(0, str(PROJECT_ROOT))

TRAIN_DIR = PROJECT_ROOT / 'data' / 'segmentation' / 'train'
MODELS_DIR = PROJECT_ROOT / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

from src.models.segmentation_detector import SegmentationDetector, TrainConfig

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
print('TRAIN_DIR:', TRAIN_DIR)

In [ ]:
model_out = MODELS_DIR / 'segmentation_unet_combined.pt'
TRAIN_EPOCHS = 300
TRAIN_BATCH_SIZE = 8
TRAIN_LR = 1e-3
TRAIN_VAL_SPLIT = 0.1
TRAIN_SEED = 42

if model_out.exists():
    print('Model already exists at', model_out, '- skipping training')
else:
    cfg = TrainConfig(
        dataset_dir=TRAIN_DIR,
        output_path=model_out,
        epochs=TRAIN_EPOCHS,
        batch_size=TRAIN_BATCH_SIZE,
        lr=TRAIN_LR,
        val_split=TRAIN_VAL_SPLIT,
        seed=TRAIN_SEED,
        device=DEVICE,
    )
    detector = SegmentationDetector(device=DEVICE)
    print('Starting training...')
    t0 = time.time()
    metrics = detector.train(cfg)
    t1 = time.time()
    print('Done. Time (s):', t1 - t0)
    print('Metrics:', metrics)